# Embed & cluster the admission-medication column

This notebook embeds the admission-medication text locally and clusters it with
KMeans, so you can draw a stratified sample for annotation.

**Pipeline:** `encode` (dense embeddings) -> L2 normalize -> KMeans

Everything runs on your machine. Your clinical text never leaves the computer.

---

### Before you start: download the model once (setup step, *outside* this notebook)

The only network access in this whole workflow is the one-time model download.
Do it once on a machine with internet, then this notebook runs fully offline:

```bash
uvx hf download sentence-transformers/all-MiniLM-L6-v2

```

This caches the weights (~80 MB) under `~/.cache/huggingface/`. If your analysis
environment is air-gapped, run the command on a connected machine and copy the
cache over. After that, the cells below never touch the network.

## 1. Import & Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
#from sklearn.decomposition import PCA
#from sklearn.metrics import silhouette_score

from sklearn.preprocessing import normalize
from sklearn.cluster import MiniBatchKMeans
from kneed import KneeLocator


In [ ]:
load_dotenv()

### 1.1. Force Offline Mode

These env vars tell the Hugging Face libraries: *use only the local cache, never
the network.* If the model is cached, it loads normally; if it isn't, you get an
error instead of a silent download. That turns "nothing hits the network" into a
guarantee you can verify.

**This must be the first cell** — the libraries read these vars at import time, so
they have to be set before any huggingface import.

In [ ]:
# Block both HF layers: the hub client and the transformers library.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

print("Offline mode set. The model must already be cached (see setup step above).")



## 2. Local Config

One place for the knobs you'll actually touch.
Fix the seeds: 
- PCA and KMeans are
otherwise non-deterministic.

### 2.1. Local Variables

In [ ]:
DATASET_PATH = f'{os.environ["REPO_PATH"]}/data/datasets/structured_extraction/medication_on_admission/medication_on_admission_final_dataset.parquet'
TEXT_COLUMN  = "meds_on_admission_cleaned"   # the medication column to embed


In [ ]:
output_data_path = f'{os.environ["REPO_PATH"]}/scripts/3_information_extraction/medications_on_admission/data'

# Creates the entire folder structure; does nothing if they already exist
os.makedirs(output_data_path, exist_ok=True)

k_range = range(2, 21, 2)



### 2.2. Embed & Cluster Config

In [ ]:
EMBEDDING_MODEL_NAME   = "all-MiniLM-L6-v2"       # a BERT model fine-tuned for sentence embeddings -> Sentence-BERT
#N_CLUSTERS   = 8                        # k for KMeans (tune separately, next step)
#MAX_TOKENS   = 512                     # this model's hard limit; longer text is truncated
#PCA_VARIANCE = 0.95                    # keep enough components to retain 95% of variance
#RANDOM_STATE = 42                      # fixes PCA + KMeans determinism
EMB_CACHE    = f'{output_data_path}/embeddings.npy'        # embeddings are expensive on 312K notes -> cache them

## 3. Load Dataset

In [ ]:
df = pd.read_parquet(DATASET_PATH)
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns")


In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df1 = df.copy()

df1['meds_on_admission_cleaned_length'] = df1['meds_on_admission_cleaned'].str.len()

df1 = df1.drop(columns = ['meds_on_admission_length'])

df1.info()

In [ ]:
df1['meds_on_admission_cleaned_length'].min()

In [ ]:
df1['meds_on_admission_cleaned_length'].max()

In [ ]:
df1[df1['meds_on_admission_cleaned_length']==5849]

In [ ]:
texts = df1[TEXT_COLUMN].fillna("").astype(str).tolist()
print(f"Notes to embed: {len(texts)}")

## 4. Embed

### 4.1. Load the model (offline)

If this cell errors, the model isn't cached yet — run the `hf download` setup step
first. If it loads cleanly with the offline vars set, that's your proof the
embedding step won't touch the network.

In [ ]:
print("Loading AI model")
print("=" * 50)

# Load model for embeddings
model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print(f"Loaded {EMBEDDING_MODEL_NAME} | embedding dim = {model.get_embedding_dimension()}")


`normalize_embeddings=True` makes each vector unit-length, so KMeans' Euclidean
distance behaves like cosine similarity — the right metric for text embeddings.

We cache to disk so you never re-embed all ~285K notes between clustering experiments.

In [ ]:
if os.path.exists(EMB_CACHE):
    print(f"Loading cached embeddings from {EMB_CACHE}")
    emb = np.load(EMB_CACHE)
else:
    emb = model.encode(
        texts,
        normalize_embeddings=True,   # L2 normalization done here
        batch_size=64,               # raise if you have GPU headroom
        show_progress_bar=True,
    ).astype(np.float32)
    np.save(EMB_CACHE, emb)
    print(f"Saved embeddings to {EMB_CACHE}")

print(f"Embedding matrix: {emb.shape}")   # (n_notes, 384) for MiniLM

### 4.2 Clustering

With embeddings computed for all notes, we now partition the population into clusters. 

The goal is **not** to discover natural structure in the data — it is to create a discretized variable that will feed the composite stratification key used to draw the annotation sample.



#### 4.2.1 Choosing K with the elbow method

We sweep K over a range and record the **inertia** (within-cluster sum of squared distances to centroids) for each value. As K grows, inertia decreases monotonically — every additional cluster can only reduce (or leave unchanged) the total distance. 

- The "elbow" is the point where the marginal gain from adding another cluster drops sharply: beyond it, we are paying in complexity without meaningful reduction in inertia.

- We use `MiniBatchKMeans` because the full `KMeans` would be prohibitively slow across a full sweep on ~285K notes. 

- `n_init=10` runs each K with 10 different initializations and keeps the best, so the elbow curve is not contaminated by unlucky starts. `random_state=42` makes the sweep reproducible.

- The elbow is detected programmatically with `KneeLocator` (`curve="convex", direction="decreasing"`) rather than by visual inspection, to remove ambiguity.


In [ ]:
# Sweep K and record inertia
embeddings = np.load(EMB_CACHE)        # cached embeddings (already L2-normalized)

inertias = []

for k in k_range:
    kmeans = MiniBatchKMeans(
        n_clusters=k,
        random_state=42,
        batch_size=4096,
        n_init=10,
    )
    kmeans.fit(embeddings)
    inertias.append(kmeans.inertia_)
    print(f"K={k:>2}: inertia={kmeans.inertia_:.0f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(k_range), inertias, marker="o")
ax.set_xlabel("K")
ax.set_ylabel("Inertia (within-cluster sum of squares)")
ax.set_title("Elbow method")
ax.set_xticks(list(k_range))
ax.grid(alpha=0.3)
plt.show()

In [ ]:

kl = KneeLocator(
    list(k_range),
    inertias,
    curve="convex",
    direction="decreasing",
)
print(f"Suggested elbow at K={kl.elbow}")

**Why k_range = range(2, 21):**

- The lower bound of 2 is the smallest meaningful K — K=1 places every note in a single cluster, which gives you the total variance of the dataset as inertia and no partition to speak of. It is not informative for the elbow.

- The upper bound of 21 (so K goes up to 20) is a pragmatic ceiling. 
    - Elbows in practice tend to appear at low K when they exist at all; extending the sweep to 30 or 50 mostly adds compute time and a long, flat tail that makes the elbow visually and algorithmically harder to locate — KneeLocator can get confused by very long post-elbow segments.

If the elbow had landed right at K=20 (the boundary), that would be a signal to extend the range — but it landed at K=8, comfortably inside the sweep, so the range was adequate. 
- A common convention in the literature is range(2, sqrt(n)) for small n, but with n=285K that heuristic is useless; a fixed practical ceiling around 20-30 is standard for datasets of this size.

#### 4.2.2. Fitting the final KMeans

Once K is fixed, we refit `MiniBatchKMeans` once with the same configuration and **attach the cluster labels to the notes DataFrame**. 
- Labels are integers in `[0, K-1]` — the numbering is arbitrary (scikit-learn convention), not an ordering.


In [ ]:
kmeans = MiniBatchKMeans(
    n_clusters=kl.elbow,
    random_state=42,
    batch_size=4096,
    n_init=10,
)

cluster_labels = kmeans.fit_predict(embeddings)

df1["cluster"] = cluster_labels

df1.head(10)

In [ ]:
np.sort(df1["cluster"].unique())

#### 4.2.3 Distribution of notes across clusters

Before moving to stratified sampling, we inspect the distribution of notes across clusters. 

A severely unbalanced partition (e.g. one cluster holding the majority of notes) is not a bug — KMeans is only a discretizer here.

In [ ]:
absolute_frequency = df1["cluster"].value_counts().sort_index()
relative_frequency = (df1["cluster"].value_counts(normalize=True).sort_index() * 100).round(2)

pd.DataFrame({
    "Absolute Frequency (number of notes per cluster)": absolute_frequency,
    "Relative Frequency (%)": relative_frequency,
})